In [1]:
import os
# os.environ["JAX_PLATFORM_NAME"] = "cpu"
os.environ["CUDA_VISIBLE_DEVICES"] = "3"
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.95"

import jax
import jax.numpy as jnp
import jax.random as jrandom
import numpy as np
import dill

In [2]:
batch_data = dill.load(open("debug.pkl", "rb"))
batch = batch_data["batch"]
rewards = batch_data["rewards"]

In [49]:
eos_token_id = 6
reset_token_id = 3
gamma = 1.0

@jax.jit
def scan_fn(carry, idx):
    pointer_correct = carry["pointer_correct"]
    actions = carry["actions"]
    target = carry["target"]
    pred_mask = carry["pred_mask"]
    last_reset_idx = carry["last_reset_idx"]
    reset_idxes = carry["reset_idxes"]
    correct_lens = carry["correct_lens"]
    curr_trial = carry["curr_trial"]

    action_match = actions[idx] == target[pointer_correct]
    is_reset = actions[idx] == reset_token_id
    is_reset_with_pred = jnp.logical_and(is_reset, pred_mask[idx])

    # Shift the pointer if the action matches the target and we're within a prediction mask
    reset_pointer = jax.lax.select(
        is_reset,
        1,
        0,
    )
    pointer_correct = jax.lax.select(
        action_match,
        pointer_correct + 1, # Increment pointer by 1 if current token matches
        reset_pointer, # Reset to 0 if it's a mistake and not a reset token, to 1 otherwise
    )

    jax.debug.print(
        "idx: {idx}, pc={pointer_correct}, trial={curr_trial}, act={act}, cl={correct_lens}",
        idx=idx,
        pointer_correct=pointer_correct,
        curr_trial=curr_trial,
        act=actions[idx],
        correct_lens=correct_lens,
    )

    # Update the last reset index to current index upon new trial
    last_reset_idx = jax.lax.select(
        is_reset_with_pred,
        idx,
        last_reset_idx,
    )

    reset_idxes = reset_idxes.at[curr_trial + 1].set(
        jax.lax.select(
            is_reset_with_pred,
            last_reset_idx,
            reset_idxes[curr_trial + 1],
        )
    )

    correct_lens = correct_lens.at[curr_trial].set(
        jnp.maximum(pointer_correct, correct_lens[curr_trial])
    )

    curr_trial = jax.lax.select(
        is_reset_with_pred,
        curr_trial + 1,
        curr_trial,
    )

    return {
        "pointer_correct": pointer_correct,
        "last_reset_idx": last_reset_idx,
        "actions": actions,
        "target": target,
        "pred_mask": pred_mask,
        "reset_idxes": reset_idxes,
        "correct_lens": correct_lens,
        "curr_trial": curr_trial,
    }, None

In [211]:
returns = np.zeros(batch["observations"].shape)

sample_i = 20
# sample_i = 10
# sample_i = 0
(pred_mask, actions, target) = (batch["pred_mask"][sample_i], batch["actions"][sample_i], batch["target"][sample_i])
(pred_mask, actions, target) = (batch_2["pred_mask"][sample_i], batch_2["actions"][sample_i], batch_2["target"][sample_i])

pointer_correct = np.array(1, dtype=int)
last_reset_idx = np.array(-1, dtype=int)
curr_trial = np.array(0, dtype=int)
reset_idxes = np.full_like(actions, fill_value=-1, dtype=int)
reset_idxes[0] = np.where(pred_mask == 1)[0][0] - 1
correct_lens = np.full_like(actions, fill_value=-1, dtype=int)
last_idx = min(np.where(pred_mask == 1)[0][-1] + 2, actions.shape[-1])

res, _ = jax.lax.scan(
    scan_fn,
    {
        "pointer_correct": pointer_correct,
        "last_reset_idx": last_reset_idx,
        "actions": actions.at[:reset_idxes[0] + 1].set(reset_token_id),
        "target": target,
        "pred_mask": pred_mask.astype(int),
        "reset_idxes": reset_idxes,
        "correct_lens": correct_lens,
        "curr_trial": curr_trial,
    },
    np.arange(last_idx),
)

reset_idxes = res["reset_idxes"]
correct_lens = res["correct_lens"]
correct_lens = np.concatenate(([0], correct_lens))

last_idx = min(np.where(pred_mask == 1)[0][-1] + 1, actions.shape[-1])
# cum_correct_lens = np.maximum.accumulate(correct_lens)
# improvements = correct_lens[1:] - cum_correct_lens[:-1]
improvements = correct_lens[1:] - correct_lens[:-1]
reset_idxes = reset_idxes.at[(np.where(reset_idxes == -1))[0][0]].set(last_idx)
trial_lengths = np.diff(reset_idxes[reset_idxes != -1])

# Update the returns array
returns[sample_i, reset_idxes[0]:last_idx] = np.repeat(
    gamma ** (
        np.arange(int(np.sum(reset_idxes != -1)) - 1)
    ) * improvements[:int(np.sum(reset_idxes != -1)) - 1],
    trial_lengths,
)

idx: 0, pc=1, trial=0, act=3, cl=[-1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1
 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1
 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1
 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1
 -1 -1 -1 -1 -1]
idx: 1, pc=1, trial=0, act=3, cl=[ 1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1
 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1
 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1
 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1
 -1 -1 -1 -1 -1]
idx: 2, pc=1, trial=0, act=3, cl=[ 1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1
 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1
 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1
 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 

In [212]:
print(target)
start_pred_idx = np.where(pred_mask)[0][0].item()
print(start_pred_idx)

[3 0 0 0 0 0 1 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6
 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6
 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6]
11


In [213]:
reset_idxes

Array([10, 17, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,
       -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,
       -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,
       -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,
       -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,
       -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],      dtype=int32)

In [214]:
correct_lens

array([ 0,  7, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,
       -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,
       -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,
       -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,
       -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,
       -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1])

In [215]:
batch["observations"][sample_i][np.where(batch["pred_mask"][sample_i])]

Array([3, 0, 0, 0, 0, 2, 4, 4, 2, 4, 2, 4, 4, 5, 4, 4, 4, 4, 4, 4, 4, 2,
       4, 4, 4, 4, 2, 4, 4, 4, 4, 4, 3, 0, 0, 0, 0, 0], dtype=int32)

In [216]:
returns[sample_i, reset_idxes[0]:last_idx]

array([7., 7., 7., 7., 7., 7., 7.])

In [217]:
reset_idxes[0], last_idx

(Array(10, dtype=int32), np.int64(17))

In [187]:
batch["observations"][sample_i]

Array([1, 1, 1, 1, 0, 2, 1, 0, 0, 0, 1, 3, 0, 0, 0, 0, 2, 4, 4, 2, 4, 2,
       4, 4, 5, 4, 4, 4, 4, 4, 4, 4, 2, 4, 4, 4, 4, 2, 4, 4, 4, 4, 4, 3,
       0, 0, 0, 0, 0, 1, 4, 5, 4, 4, 2, 4, 4, 4, 4, 4, 2, 4, 4, 2, 2, 4,
       3, 0, 0, 2, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 2, 4, 4, 4, 2, 4, 4, 4,
       2, 5, 4, 2, 4, 4, 4, 4, 4, 4, 4, 2, 2], dtype=int32)

In [188]:
batch["pred_mask"][sample_i]

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], dtype=int32)

In [189]:
batch["actions"][sample_i]

Array([0, 2, 1, 1, 0, 0, 2, 1, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 2, 0, 2,
       0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 3,
       0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 2, 0, 0, 0, 0, 0, 2, 0, 0, 2, 2, 0,
       3, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 2, 0, 0, 0,
       2, 1, 0, 2, 0, 0, 0, 0, 0, 0, 0, 2, 2], dtype=int32)

In [127]:
batch["actions"][sample_i][np.where(pred_mask)[0]]

Array([0, 0, 0, 0, 0, 2, 0, 0, 2, 0, 2, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 2,
       0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 3, 0, 0, 0, 0, 0], dtype=int32)

In [16]:
metastable_aug_p = 0.1

In [ ]:
def _sample_reset_idxes(obss, rlengths, rngs):
    reset_idxes = jnp.where(
        obss == reset_token_id,
        size=len(obss),
        fill_value=-1,
    )[0]
    first_reset_idx = reset_idxes[0]

    reset_idxes = jax.lax.select(
        reset_idxes >= first_reset_idx + rlengths,
        jnp.full_like(reset_idxes, fill_value=-1),
        reset_idxes,
    )

    logits = jax.lax.select(
        reset_idxes == -1,
        -jnp.inf,
        0.0,
    )
    reset_idx_i = jrandom.categorical(rngs, logits[1:])
    reset_idx = jax.lax.select(
        jnp.logical_or(
            reset_idxes[reset_idx_i + 1] == -1,
            jrandom.uniform(rngs) < metastable_aug_p,
        ),
        reset_idxes[0],
        reset_idxes[reset_idx_i + 1],
    )
    return reset_idx, first_reset_idx

def _augment_sample(
    out_obss,
    out_acts,
    out_masks,
    out_rets,
    first_reset_idx,
    reset_idx,
):
    delta = reset_idx - first_reset_idx
    delta = jax.lax.select(
        delta <= 0,
        len(out_obss),
        -delta,
    )

    # Given [X_1, ..., X_T, <EQUAL>, ...]
    # Want a mask where <EQUAL> and onward are set to 0.
    first_reset_mask = jnp.zeros_like(out_obss)
    first_reset_mask = first_reset_mask.at[first_reset_idx].set(1)
    question_mask = 1 - jnp.cumsum(first_reset_mask)

    # Given [X_1, ..., X_T, <EQUAL>, ..., <EQUAL_N>, ...]
    # Want a mask where <EQUAL_N> and onward are set to 1.
    reset_mask = jnp.zeros_like(out_obss)
    reset_mask = reset_mask.at[reset_idx].set(1)
    reset_mask = jnp.cumsum(reset_mask)

    # Let delta = position of <EQUAL_N> - position of <EQUAL>.
    # Want a mask where the last delta entries are set to 1
    # If delta = 0, then all entries are set to 0.
    eos_mask = jnp.zeros(out_obss.shape[0])
    eos_mask = jax.lax.select(
        delta == len(out_obss),
        eos_mask,
        eos_mask.at[delta].set(1),
    )
    eos_mask = jnp.cumsum(eos_mask)

    out_obss = question_mask * out_obss + jnp.roll(reset_mask * out_obss, delta)
    out_obss = (1 - eos_mask) * out_obss + eos_mask * eos_token_id

    out_acts = question_mask * out_acts + jnp.roll(reset_mask * out_acts, jax.lax.select(delta < 0, delta - 1, delta))

    out_masks = question_mask * out_masks + jnp.roll(reset_mask * out_masks, delta)
    out_masks = (1 - eos_mask) * out_masks

    out_rets = question_mask * out_rets + jnp.roll(reset_mask * out_rets, delta)
    out_rets = (1 - eos_mask) * out_rets

    return (
        out_obss.astype(int),
        out_acts.astype(int),
        out_masks.astype(int),
        out_rets.astype(int),
    )

def _identity(
    out_obss,
    out_acts,
    out_masks,
    out_rets,
    first_reset_idx,
    reset_idx,
):
    return (
        out_obss.astype(int),
        out_acts.astype(int),
        out_masks.astype(int),
        out_rets.astype(int),
    )

def _augment(iter_i, state):
    obss = state["observations"][iter_i]
    acts = state["actions"][iter_i]
    masks = state["pred_mask"][iter_i]
    rets = state["returns"][iter_i]
    first_reset_idx = state["first_reset_idx"][iter_i]
    reset_idx = state["reset_idx"][iter_i]
    success = state["successes"][iter_i]
    out_obss = jnp.copy(obss)
    out_acts = jnp.copy(acts)
    out_masks = jnp.copy(masks)
    out_rets = jnp.copy(rets)
    (
        out_obss,
        out_acts,
        out_masks,
        out_rets,
    ) = jax.lax.cond(
        success,
        _augment_sample,
        _identity,
        out_obss,
        out_acts,
        out_masks,
        out_rets,
        first_reset_idx,
        reset_idx,
    )

    return {
        "observations": state["observations"].at[iter_i].set(out_obss),
        "actions": state["actions"].at[iter_i].set(out_acts),
        "pred_mask": state["pred_mask"].at[iter_i].set(out_masks),
        "returns": state["returns"].at[iter_i].set(out_rets),
        "first_reset_idx": state["first_reset_idx"],
        "reset_idx": state["reset_idx"],
        "successes": state["successes"],
    }
sample_reset_idxes = jax.vmap(_sample_reset_idxes)
augment = jax.jit(_augment)

In [219]:
curr_rng = jrandom.PRNGKey(42)
aug_rngs = jrandom.split(curr_rng, len(batch["observations"]))
                
reset_idx, first_reset_idx = sample_reset_idxes(
    batch["observations"],
    np.sum(batch["pred_mask"], axis=-1),
    aug_rngs,
)

batch["returns"] = returns

aug_data = jax.lax.fori_loop(
    sample_i,
    sample_i + 1,
    augment,
    {
        "observations": batch["observations"],
        "actions": batch["actions"],
        "pred_mask": batch["pred_mask"],
        "returns": batch["returns"],
        "first_reset_idx": first_reset_idx,
        "reset_idx": reset_idx,
        "successes": rewards > 0.0,
    }
)

TypeError: select() missing 1 required positional argument: 'on_false'

In [220]:
sample_i = 20

In [206]:
aug_data["actions"][sample_i]

Array([0, 2, 1, 1, 0, 0, 2, 1, 0, 0, 3, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 2,
       0, 0, 0, 0, 0, 2, 0, 0, 2, 2, 0, 3, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 2, 0, 0, 0, 2, 0, 0, 0, 2, 1, 0, 2, 0, 0, 0, 0, 0, 0, 0,
       2, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], dtype=int32)

In [207]:
aug_data["observations"][sample_i]

Array([1, 1, 1, 1, 0, 2, 1, 0, 0, 0, 1, 3, 0, 0, 0, 0, 0, 1, 4, 5, 4, 4,
       2, 4, 4, 4, 4, 4, 2, 4, 4, 2, 2, 4, 3, 0, 0, 2, 4, 4, 4, 4, 4, 4,
       4, 4, 4, 4, 2, 4, 4, 4, 2, 4, 4, 4, 2, 5, 4, 2, 4, 4, 4, 4, 4, 4,
       4, 2, 2, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6,
       6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6], dtype=int32)

In [208]:
aug_data["pred_mask"][sample_i]

Array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], dtype=int32)

In [209]:
aug_data["pred_mask"][sample_i] - batch["pred_mask"][sample_i], jnp.sum(aug_data["pred_mask"][sample_i]), jnp.sum(batch["pred_mask"][sample_i])

(Array([ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
        -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,
        -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],      dtype=int32),
 Array(6, dtype=int32),
 Array(38, dtype=int32))

In [24]:
assert 0

AssertionError: 

In [210]:
import copy

batch_2 = copy.deepcopy(batch)
(
    batch_2["observations"],
    batch_2["actions"],
    batch_2["pred_mask"],
    batch_2["returns"],
) = (
    aug_data["observations"],
    aug_data["actions"],
    aug_data["pred_mask"],
    aug_data["returns"],
)

In [110]:
batch_2["observations"][sample_i]

Array([1, 1, 1, 1, 0, 2, 1, 0, 0, 0, 1, 3, 0, 0, 0, 0, 0, 1, 4, 5, 4, 4,
       2, 4, 4, 4, 4, 4, 2, 4, 4, 2, 2, 4, 3, 0, 0, 2, 4, 4, 4, 4, 4, 4,
       4, 4, 4, 4, 2, 4, 4, 4, 2, 4, 4, 4, 2, 5, 4, 2, 4, 4, 4, 4, 4, 4,
       4, 2, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], dtype=int32)

In [111]:
batch["observations"][sample_i]

Array([1, 1, 1, 1, 0, 2, 1, 0, 0, 0, 1, 3, 0, 0, 0, 0, 2, 4, 4, 2, 4, 2,
       4, 4, 5, 4, 4, 4, 4, 4, 4, 4, 2, 4, 4, 4, 4, 2, 4, 4, 4, 4, 4, 3,
       0, 0, 0, 0, 0, 1, 4, 5, 4, 4, 2, 4, 4, 4, 4, 4, 2, 4, 4, 2, 2, 4,
       3, 0, 0, 2, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 2, 4, 4, 4, 2, 4, 4, 4,
       2, 5, 4, 2, 4, 4, 4, 4, 4, 4, 4, 2, 2], dtype=int32)

In [112]:
batch_2["actions"][sample_i][np.where(batch_2["pred_mask"][sample_i])]

Array([3, 0, 0, 0, 0, 0], dtype=int32)

In [113]:
batch["actions"][sample_i][np.where(batch["pred_mask"][sample_i])]

Array([0, 0, 0, 0, 0, 2, 0, 0, 2, 0, 2, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 2,
       0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 3, 0, 0, 0, 0, 0], dtype=int32)

In [114]:
batch_2["actions"][sample_i][11:]

Array([3, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 2, 0, 0, 0, 0, 0, 2, 0, 0, 2, 2,
       0, 3, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 2, 0, 0,
       0, 2, 1, 0, 2, 0, 0, 0, 0, 0, 0, 0, 2, 2, 0, 2, 1, 1, 0, 0, 2, 1,
       0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 2, 0, 2, 0, 0, 1, 0, 0, 0, 0, 0,
       0, 0], dtype=int32)

In [115]:
batch["actions"][sample_i][11:]

Array([0, 0, 0, 0, 0, 2, 0, 0, 2, 0, 2, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 2,
       0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 2,
       0, 0, 0, 0, 0, 2, 0, 0, 2, 2, 0, 3, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 2, 0, 0, 0, 2, 0, 0, 0, 2, 1, 0, 2, 0, 0, 0, 0, 0, 0, 0,
       2, 2], dtype=int32)